# RST — Processed Dataset Visualization

Visual inspection of the **processed** `.npz` datasets produced by `build_dataset.py`.

Each sample is a `(96, 1024)` spectrogram — 6 observations of 16 time bins stacked vertically.
This notebook un-stacks them and renders **turboSETI-style waterfall plots**
with the 6 observations displayed as separate panels.

> **Observation order:** ON₁ → OFF₁ → ON₂ → OFF₂ → ON₃ → OFF₃ (top → bottom)

In [ ]:
%matplotlib inline

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from src.data.preprocessing import normalize_robust

# ── Style ─────────────────────────────────────────────────────
FONTSIZE = 14
FONT = {'family': 'DejaVu Sans', 'size': FONTSIZE}
matplotlib.rc('font', **FONT)

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': '#0e1117',
    'axes.facecolor': '#161b22',
    'savefig.facecolor': '#0e1117',
    'text.color': '#e6edf3',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'axes.edgecolor': '#30363d',
    'grid.color': '#21262d',
    'grid.alpha': 0.5,
})

CMAP = 'viridis'
OBS_LABELS  = ['ON₁', 'OFF₁', 'ON₂', 'OFF₂', 'ON₃', 'OFF₃']
ON_COLOR    = '#3fb950'
OFF_COLOR   = '#f85149'
OBS_COLORS  = [ON_COLOR, OFF_COLOR] * 3
TCHANS_OBS  = 16  # time bins per observation

print('✓ Setup complete')

---
## 1. Load Processed Dataset

Load a `.npz` file produced by `build_dataset.py`.
Each file contains:
- `spectrograms` — shape `(N, 96, 1024)`, raw stacked cadences
- `labels` — shape `(N,)`, `1 = True (ETI)`, `0 = False (RFI/noise)`

In [ ]:
# ===== ⚠️  EDIT HERE  =====
DATASET_PATH = '../data/processed/train.npz'
N_SAMPLES    = 5          # number of samples to visualize per class
SEED         = 42         # for reproducible random selection
# ===========================

data = np.load(DATASET_PATH, allow_pickle=True)
spectrograms = data['spectrograms']   # (N, 96, 1024)
labels       = data['labels']          # (N,)

n_true  = int(labels.sum())
n_false = len(labels) - n_true

print(f'Dataset loaded: {DATASET_PATH}')
print(f'  Shape: {spectrograms.shape}')
print(f'  True:  {n_true}  |  False: {n_false}')

In [ ]:
# ── Separate and sample ──────────────────────────────────────
rng = np.random.default_rng(SEED)

true_idx  = np.where(labels == 1)[0]
false_idx = np.where(labels == 0)[0]

sel_true  = rng.choice(true_idx,  size=min(N_SAMPLES, len(true_idx)),  replace=False)
sel_false = rng.choice(false_idx, size=min(N_SAMPLES, len(false_idx)), replace=False)

print(f'Selected {len(sel_true)} True + {len(sel_false)} False samples for visualization')

---
## 2. Plotting Utilities

In [ ]:
def split_observations(spec, tchans=TCHANS_OBS):
    """Split a (96, W) spectrogram back into 6 individual (16, W) observations."""
    n_obs = spec.shape[0] // tchans
    return [spec[i * tchans : (i + 1) * tchans, :] for i in range(n_obs)]


def plot_cadence_turboseti(spec, title='', normalize=True):
    """
    Plot a processed (96, 1024) sample as 6 vertically-stacked
    turboSETI-style waterfall panels.

    Parameters
    ----------
    spec : ndarray (96, 1024)
        Stacked spectrogram from the processed dataset.
    title : str
        Figure title.
    normalize : bool
        If True, apply normalize_robust before plotting.
    """
    if normalize:
        spec = normalize_robust(spec.copy())

    observations = split_observations(spec)
    n_obs = len(observations)

    # Common intensity scale across the 6 panels
    vmin = np.percentile(spec, 2)
    vmax = np.percentile(spec, 98)

    fig, axes = plt.subplots(
        n_obs, 1,
        figsize=(12, 1.3 * n_obs),
        sharex=True, sharey=True,
    )

    for i, ax in enumerate(axes):
        im = ax.imshow(
            observations[i], aspect='auto', cmap=CMAP,
            vmin=vmin, vmax=vmax, origin='upper',
            interpolation='nearest', rasterized=True,
        )

        # ON/OFF label inside each panel
        ax.text(
            0.015, 0.5, OBS_LABELS[i],
            transform=ax.transAxes, va='center', ha='left',
            fontsize=11, fontweight='bold', color=OBS_COLORS[i],
            bbox=dict(facecolor='black', alpha=0.6,
                      edgecolor='none', boxstyle='round,pad=0.3'),
        )

        ax.set_ylabel('Time', fontsize=10)
        ax.set_yticks([])

        if i < n_obs - 1:
            ax.tick_params(labelbottom=False)

    axes[-1].set_xlabel('Frequency Channel')
    if title:
        axes[0].set_title(title, fontweight='bold', fontsize=14, pad=8)

    fig.colorbar(
        im, ax=axes, shrink=0.75, pad=0.02,
        label='Normalized Intensity',
    )
    plt.subplots_adjust(hspace=0, wspace=0)

    return fig

---
## 3. TRUE Samples (ETI)

Signal should appear **only in the ON windows** (ON₁, ON₂, ON₃).

In [ ]:
for i, idx in enumerate(sel_true):
    spec = spectrograms[idx]
    fig = plot_cadence_turboseti(
        spec,
        title=f'TRUE #{i}  (dataset index {idx})',
    )
    plt.show()

---
## 4. FALSE Samples (RFI / Noise)

RFI signals appear in **all 6 observations** (both ON and OFF),
or no signal at all (pure background).

In [ ]:
for i, idx in enumerate(sel_false):
    spec = spectrograms[idx]
    fig = plot_cadence_turboseti(
        spec,
        title=f'FALSE #{i}  (dataset index {idx})',
    )
    plt.show()

---
## 5. Side-by-Side Comparison (TRUE vs FALSE)

Each row shows a pair of samples for quick comparison.

In [ ]:
import matplotlib.gridspec as gridspec

n_pairs = min(len(sel_true), len(sel_false), 3)

for p in range(n_pairs):
    spec_t = normalize_robust(spectrograms[sel_true[p]].copy())
    spec_f = normalize_robust(spectrograms[sel_false[p]].copy())

    obs_t = split_observations(spec_t)
    obs_f = split_observations(spec_f)

    fig = plt.figure(figsize=(22, 9))
    gs = gridspec.GridSpec(6, 2, figure=fig, hspace=0, wspace=0.12)

    axes_t = [fig.add_subplot(gs[row, 0]) for row in range(6)]
    axes_f = [fig.add_subplot(gs[row, 1]) for row in range(6)]

    for col_axes, obs_list, label, color in [
        (axes_t, obs_t, f'TRUE  (idx {sel_true[p]})',  ON_COLOR),
        (axes_f, obs_f, f'FALSE (idx {sel_false[p]})', OFF_COLOR),
    ]:
        all_data = np.vstack(obs_list)
        vmin = np.percentile(all_data, 2)
        vmax = np.percentile(all_data, 98)

        for obs_i, ax in enumerate(col_axes):
            im = ax.imshow(
                obs_list[obs_i], aspect='auto', cmap=CMAP,
                vmin=vmin, vmax=vmax, origin='upper',
                interpolation='nearest', rasterized=True,
            )
            ax.text(
                0.015, 0.5, OBS_LABELS[obs_i],
                transform=ax.transAxes, va='center', ha='left',
                fontsize=9, fontweight='bold', color=OBS_COLORS[obs_i],
                bbox=dict(facecolor='black', alpha=0.6,
                          edgecolor='none', boxstyle='round,pad=0.2'),
            )
            ax.set_yticks([])
            if obs_i < 5:
                ax.tick_params(labelbottom=False)
            if obs_i == 0:
                ax.set_title(label, fontweight='bold', fontsize=12,
                             pad=6, color=color)

        col_axes[-1].set_xlabel('Frequency Channel')

    fig.colorbar(im, ax=axes_f, shrink=0.75, pad=0.03,
                 label='Normalized Intensity')
    plt.show()

---
## 6. Intensity Statistics

Distribution of pixel intensities before and after `normalize_robust`.

In [ ]:
n_stats = min(200, len(spectrograms))
stat_idx = rng.choice(len(spectrograms), n_stats, replace=False)

raw_vals  = spectrograms[stat_idx].flatten()
norm_vals = np.array([normalize_robust(spectrograms[i].copy()) for i in stat_idx]).flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Raw
axes[0].hist(raw_vals, bins=120, alpha=0.85, color='#79dafa',
             density=True, edgecolor='none')
axes[0].set_title('Raw Intensity Distribution', fontweight='bold')
axes[0].set_xlabel('Intensity')
axes[0].set_ylabel('Density')
axes[0].axvline(raw_vals.mean(), color='#f0883e', ls='--', lw=1.5,
                label=f'μ = {raw_vals.mean():.2e}')
axes[0].legend(fontsize=10)

# Normalized
axes[1].hist(norm_vals, bins=120, alpha=0.85, color='#79dafa',
             density=True, edgecolor='none')
axes[1].set_title('Normalized (normalize_robust)', fontweight='bold')
axes[1].set_xlabel('Normalized value')
axes[1].set_ylabel('Density')
axes[1].axvline(0, color='#58a6ff', ls='--', lw=1, alpha=0.7, label='μ = 0')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f'Raw  → min={raw_vals.min():.2e}, max={raw_vals.max():.2e}, '
      f'mean={raw_vals.mean():.2e}, std={raw_vals.std():.2e}')
print(f'Norm → min={norm_vals.min():.3f}, max={norm_vals.max():.3f}, '
      f'mean={norm_vals.mean():.3f}, std={norm_vals.std():.3f}')